# Original ALBEF vs FC vs Gated vs Mean vs Residual vs Transformer heatmaps

This notebook extends the original-vs-FC comparison notebook to all multi-view fusion experiments while preserving the same comparison protocol.

**Main PDF columns**

1. Original CXR + VinDr GT box
2. Original single-view ALBEF overlay
3. FC fusion overlay
4. Gated fusion overlay
5. Mean fusion overlay
6. Residual fusion overlay
7. Transformer fusion overlay

The comparison is restricted to the **same canonical 100 BioViL-T image-label pairs** (50 Cardiomegaly + 50 Pleural effusion) so every method is assessed on exactly the same qualitative cases.

A second PDF provides branch-level details for each fusion method.

In [ ]:
from pathlib import Path
import math
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 240)

TARGET_LABELS = ['Cardiomegaly', 'Pleural effusion']
EXPECTED_PAIR_COUNT = 100
EXPECTED_PER_LABEL = 50
EXPECTED_IMAGE_RES = 256
CASES_PER_PAGE = 4
OVERLAY_ALPHA = 0.50
CMAP_NAME = 'magma'
FIG_DPI = 150
GENERATE_BRANCH_DETAIL_PDF = True

## Paths — edit these before running

In [ ]:
# Heatmap output directories
ORIGINAL_ALBEF_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/original_albef_itc_margin_gradcam_test_best_cardio'
)
FUSED_FC_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/fc_fusion_itc_margin_gradcam_test_best_cardio'
)
GATED_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/gated_fusion_itc_margin_gradcam'
)
MEAN_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/mean_fusion_itc_margin_gradcam'
)
RESIDUAL_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/residual_fusion_itc_margin_gradcam'
)
TRANSFORMER_DIR = Path(
    '/home/vault/iwi5/iwi5362h/outputs/transformer_fusion_itc_margin_gradcam'
)

# Canonical 100-pair BioViL-T manifest used in your earlier comparisons.
BIOVIL_OUTPUT_DIR = Path(
    '/home/vault/iwi5/iwi5362h/results/biovil_t_phrase_grounding/same_50_albef_margin'
)
BIOVIL_MANIFEST_CSV = BIOVIL_OUTPUT_DIR / 'manifest.csv'

IMAGES_ROOT = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test')
ANNOTATIONS_CSV = Path(
    '/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv'
)
IMAGE_METADATA_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv')

VIS_OUTPUT_DIR = Path(
    '/home/vault/iwi5/iwi5362h/results/visualization/'
    'original_fc_gated_mean_residual_transformer_heatmaps'
)
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_FILES = {
    'original': ORIGINAL_ALBEF_DIR / 'itc_margin_gradcam_index.csv',
    'fc': FUSED_FC_DIR / 'fc_fusion_itc_margin_gradcam_index.csv',
    'gated': GATED_DIR / 'gated_fusion_itc_margin_gradcam_index.csv',
    'mean': MEAN_DIR / 'mean_fusion_itc_margin_gradcam_index.csv',
    'residual': RESIDUAL_DIR / 'residual_fusion_itc_margin_gradcam_index.csv',
    'transformer': TRANSFORMER_DIR / 'transformer_fusion_itc_margin_gradcam_index.csv',
}
METHOD_DIRS = {
    'original': ORIGINAL_ALBEF_DIR,
    'fc': FUSED_FC_DIR,
    'gated': GATED_DIR,
    'mean': MEAN_DIR,
    'residual': RESIDUAL_DIR,
    'transformer': TRANSFORMER_DIR,
}

for path in [
    *METHOD_DIRS.values(), BIOVIL_MANIFEST_CSV,
    IMAGES_ROOT, ANNOTATIONS_CSV, IMAGE_METADATA_CSV,
]:
    if not path.exists():
        raise FileNotFoundError(path)

for name, path in INDEX_FILES.items():
    print(f'{name:11s}: {path} | exists={path.exists()}')
print('Outputs:', VIS_OUTPUT_DIR)

## Common loading and visualization helpers

In [ ]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def as_2d_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    return np.asarray(value, dtype=np.float32).squeeze()


def normalize_label(label):
    return str(label).strip().casefold()


def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(f'Cannot find {purpose}. Available columns: {list(df.columns)}')


def resolve_label_key(payload, requested_label):
    lookup = {
        normalize_label(key): key
        for key in payload.keys()
        if str(key) != '__metadata__'
    }
    key = lookup.get(normalize_label(requested_label))
    if key is None:
        available = [k for k in payload.keys() if str(k) != '__metadata__']
        raise KeyError(f'Label {requested_label!r} not present. Available: {available}')
    return key


def find_payload_path(base_dir, image_id):
    candidates = [
        base_dir / f'{image_id}.pt',
        base_dir / 'maps' / f'{image_id}.pt',
    ]
    for path in candidates:
        if path.is_file():
            return path
    matches = list(base_dir.rglob(f'{image_id}.pt'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not find {image_id}.pt under {base_dir}')


def build_payload_index(base_dir, preferred_index_csv):
    if preferred_index_csv.is_file():
        df = pd.read_csv(preferred_index_csv)
        if 'image_id' not in df.columns:
            raise KeyError(f'{preferred_index_csv} has no image_id column')
        df['image_id'] = df['image_id'].astype(str)

        if 'heatmap_path' in df.columns:
            def resolve_saved_path(saved_path):
                saved = Path(str(saved_path))
                if saved.is_file():
                    return saved
                for candidate in [base_dir / saved.name, base_dir / 'maps' / saved.name]:
                    if candidate.is_file():
                        return candidate
                return find_payload_path(base_dir, saved.stem)
            df['payload_path'] = df['heatmap_path'].map(resolve_saved_path)
        else:
            df['payload_path'] = df['image_id'].map(lambda x: find_payload_path(base_dir, x))
        return df[['image_id', 'payload_path']].drop_duplicates('image_id').reset_index(drop=True)

    payloads = sorted(base_dir.rglob('*.pt'))
    if not payloads:
        raise FileNotFoundError(f'No .pt payloads found under {base_dir}')
    return pd.DataFrame({
        'image_id': [p.stem for p in payloads],
        'payload_path': payloads,
    }).drop_duplicates('image_id').reset_index(drop=True)


def load_image(image_id):
    with Image.open(IMAGES_ROOT / f'{image_id}.png') as handle:
        return handle.convert('RGB')


def make_overlay(image, vis_map, alpha=OVERLAY_ALPHA):
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    heatmap = as_2d_numpy(vis_map)
    if heatmap.shape != rgb.shape[:2]:
        raise ValueError(f'Image/map mismatch: image={rgb.shape[:2]}, map={heatmap.shape}')
    color = colormaps[CMAP_NAME](np.clip(heatmap, 0, 1))[..., :3]
    alpha_map = (alpha * np.clip(heatmap, 0, 1))[..., None]
    return np.clip((1 - alpha_map) * rgb + alpha_map * color, 0, 1)

## Load VinDr boxes and original dimensions

In [ ]:
annotations_df = pd.read_csv(ANNOTATIONS_CSV)
metadata_df = pd.read_csv(IMAGE_METADATA_CSV)

ann_image_col = choose_column(annotations_df, ['image_id', 'imageid', 'image_name'], 'annotation image id')
ann_label_col = choose_column(annotations_df, ['class_name', 'label', 'finding_name'], 'annotation label')
ann_xmin_col = choose_column(annotations_df, ['x_min', 'xmin', 'x1'], 'annotation x_min')
ann_ymin_col = choose_column(annotations_df, ['y_min', 'ymin', 'y1'], 'annotation y_min')
ann_xmax_col = choose_column(annotations_df, ['x_max', 'xmax', 'x2'], 'annotation x_max')
ann_ymax_col = choose_column(annotations_df, ['y_max', 'ymax', 'y2'], 'annotation y_max')

meta_image_col = choose_column(metadata_df, ['image_id', 'imageid', 'image_name'], 'metadata image id')
meta_width_col = choose_column(metadata_df, ['width', 'original_width', 'w'], 'metadata width')
meta_height_col = choose_column(metadata_df, ['height', 'original_height', 'h'], 'metadata height')

annotations_df[ann_image_col] = annotations_df[ann_image_col].astype(str)
annotations_df[ann_label_col] = annotations_df[ann_label_col].astype(str)
metadata_df[meta_image_col] = metadata_df[meta_image_col].astype(str)

metadata_lookup = {
    str(row[meta_image_col]): {
        'width': float(row[meta_width_col]),
        'height': float(row[meta_height_col]),
    }
    for _, row in metadata_df.iterrows()
}


def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[
        (annotations_df[ann_image_col] == str(image_id))
        & (annotations_df[ann_label_col].map(normalize_label) == normalize_label(label))
    ]
    if subset.empty or str(image_id) not in metadata_lookup:
        return []
    original = metadata_lookup[str(image_id)]
    sx = displayed_size[0] / original['width']
    sy = displayed_size[1] / original['height']
    boxes = []
    for _, row in subset.iterrows():
        boxes.append({
            'x': float(row[ann_xmin_col]) * sx,
            'y': float(row[ann_ymin_col]) * sy,
            'w': (float(row[ann_xmax_col]) - float(row[ann_xmin_col])) * sx,
            'h': (float(row[ann_ymax_col]) - float(row[ann_ymin_col])) * sy,
        })
    return boxes


def draw_boxes(ax, boxes, color='lime', lw=1.6):
    for box in boxes:
        ax.add_patch(Rectangle(
            (box['x'], box['y']), box['w'], box['h'],
            fill=False, edgecolor=color, linewidth=lw,
        ))

## Load all six heatmap result sets

In [ ]:
indices = {
    method: build_payload_index(METHOD_DIRS[method], INDEX_FILES[method])
    for method in METHOD_DIRS
}
lookups = {
    method: dict(zip(df['image_id'], df['payload_path']))
    for method, df in indices.items()
}

common_image_ids = set(lookups['original'])
for method in ['fc', 'gated', 'mean', 'residual', 'transformer']:
    common_image_ids &= set(lookups[method])
common_image_ids = sorted(common_image_ids)

for method, df in indices.items():
    print(f'{method:11s} payloads: {len(df)}')
print('Images common to all six methods:', len(common_image_ids))

## Build full comparison table

In [ ]:
rows = []
for image_id in common_image_ids:
    payloads = {
        method: safe_torch_load(lookups[method][image_id])
        for method in lookups
    }

    for requested_label in TARGET_LABELS:
        try:
            label_keys = {
                method: resolve_label_key(payload, requested_label)
                for method, payload in payloads.items()
            }
        except KeyError:
            continue

        items = {
            method: payloads[method][label_keys[method]]
            for method in payloads
        }
        gt_values = [float(item.get('ground_truth', np.nan)) for item in items.values()]
        finite_gt = [x for x in gt_values if np.isfinite(x)]
        gt = finite_gt[0] if finite_gt else np.nan

        row = {
            'image_id': str(image_id),
            'label': requested_label,
            '_label_norm': normalize_label(requested_label),
            'ground_truth': gt,
        }
        for method in lookups:
            row[f'{method}_path'] = Path(lookups[method][image_id])
            row[f'{method}_label_key'] = label_keys[method]
            row[f'{method}_probability'] = float(items[method].get('positive_probability', np.nan))
            row[f'{method}_margin'] = float(items[method].get('margin', np.nan))

        # Fusion-specific diagnostics saved by the new extractors.
        gated_weights = items['gated'].get('view_weights', {})
        row['gated_weight_original'] = gated_weights.get('original', np.nan)
        row['gated_weight_lung'] = gated_weights.get('lung', np.nan)
        row['gated_weight_heart'] = gated_weights.get('heart', np.nan)
        residual_gates = items['residual'].get('residual_gates', {})
        row['residual_gate_lung'] = residual_gates.get('lung', np.nan)
        row['residual_gate_heart'] = residual_gates.get('heart', np.nan)
        rows.append(row)

comparison_df = pd.DataFrame(rows)
if comparison_df.empty:
    raise ValueError('No common image-label cases were found across all six methods.')

print('All common image-label cases:', len(comparison_df))
display(comparison_df.groupby('label').size().rename('pairs').to_frame())

## Restrict to the exact canonical 100 BioViL-T image-label pairs

In [ ]:
if not BIOVIL_MANIFEST_CSV.is_file():
    raise FileNotFoundError(BIOVIL_MANIFEST_CSV)

biovil_manifest = pd.read_csv(BIOVIL_MANIFEST_CSV)
required = {'image_id', 'label', 'heatmap_path'}
missing = required - set(biovil_manifest.columns)
if missing:
    raise KeyError(f'BioViL-T manifest missing columns: {sorted(missing)}')

biovil_manifest['image_id'] = biovil_manifest['image_id'].astype(str)
biovil_manifest['_label_norm'] = biovil_manifest['label'].map(normalize_label)
target_norms = {normalize_label(label) for label in TARGET_LABELS}
biovil_manifest = biovil_manifest[
    biovil_manifest['_label_norm'].isin(target_norms)
].copy()

if biovil_manifest.duplicated(['image_id', '_label_norm']).any():
    raise ValueError('Duplicate image-label pairs in BioViL-T manifest')

label_order = {normalize_label(label): i for i, label in enumerate(TARGET_LABELS)}
canonical_pairs = (
    biovil_manifest.assign(_label_order=biovil_manifest['_label_norm'].map(label_order))
    .sort_values(['_label_order'], kind='stable')
    [['image_id', 'label', '_label_norm']]
    .reset_index(drop=True)
)

counts = canonical_pairs['_label_norm'].value_counts().reindex(
    [normalize_label(x) for x in TARGET_LABELS], fill_value=0
)
print('Canonical image-label pairs:', len(canonical_pairs))
print('Unique CXRs:               ', canonical_pairs['image_id'].nunique())
display(counts.rename('pairs').to_frame())

if len(canonical_pairs) != EXPECTED_PAIR_COUNT:
    raise ValueError(
        f'Expected {EXPECTED_PAIR_COUNT} canonical image-label pairs, found {len(canonical_pairs)}'
    )
if not (counts == EXPECTED_PER_LABEL).all():
    raise ValueError(f'Expected {EXPECTED_PER_LABEL} per label, got {counts.to_dict()}')

# Left merge from canonical_pairs deliberately preserves the exact BioViL order.
gallery_df = canonical_pairs.merge(
    comparison_df.drop(columns=['label']),
    on=['image_id', '_label_norm'],
    how='left',
    validate='one_to_one',
)

required_path_cols = [f'{m}_path' for m in METHOD_DIRS]
missing_rows = gallery_df[gallery_df[required_path_cols].isna().any(axis=1)]
if not missing_rows.empty:
    raise ValueError(
        f'{len(missing_rows)} canonical cases are missing from one or more methods. '
        f'Examples:\n{missing_rows[["image_id", "label"]].head(10)}'
    )

# Keep the canonical manifest label for display.
gallery_df = gallery_df.reset_index(drop=True)
comparison_df = gallery_df.copy()

print('Final comparison cases:', len(gallery_df))
display(gallery_df.head(10))

## Map access helpers

In [ ]:
def get_original_maps(row):
    payload = safe_torch_load(row.original_path)
    item = payload[row.original_label_key]
    return {
        'vis_up': as_2d_numpy(item['cam_vis_up']),
        'probability': float(item.get('positive_probability', np.nan)),
        'margin': float(item.get('margin', np.nan)),
    }


def get_fusion_maps(row, method):
    payload = safe_torch_load(getattr(row, f'{method}_path'))
    item = payload[getattr(row, f'{method}_label_key')]
    result = {
        'fused_vis_up': as_2d_numpy(item['fused']['cam_vis_up']),
        'original_branch_vis_up': as_2d_numpy(item['original']['cam_vis_up']),
        'lung_branch_vis_up': as_2d_numpy(item['lung']['cam_vis_up']),
        'heart_branch_vis_up': as_2d_numpy(item['heart']['cam_vis_up']),
        'probability': float(item.get('positive_probability', np.nan)),
        'margin': float(item.get('margin', np.nan)),
    }
    if method == 'gated':
        result['view_weights'] = item.get('view_weights', {})
    if method == 'residual':
        result['residual_gates'] = item.get('residual_gates', {})
    return result

## Inspect one case

In [ ]:
CASE_INDEX = 0
row = gallery_df.iloc[CASE_INDEX]
image = load_image(row.image_id)
boxes = get_boxes(row.image_id, row.label, image.size)
orig = get_original_maps(row)
method_maps = {m: get_fusion_maps(row, m) for m in ['fc','gated','mean','residual','transformer']}

fig, axes = plt.subplots(1, 7, figsize=(25, 4), dpi=FIG_DPI)
axes[0].imshow(image); draw_boxes(axes[0], boxes)
axes[0].set_title(f'{row.image_id}\n{row.label} | GT={row.ground_truth}')
axes[1].imshow(make_overlay(image, orig['vis_up'])); draw_boxes(axes[1], boxes)
axes[1].set_title(f'Original ALBEF\np={orig["probability"]:.3f}')
for ax, method, title in zip(
    axes[2:],
    ['fc','gated','mean','residual','transformer'],
    ['FC','Gated','Mean','Residual','Transformer'],
):
    maps = method_maps[method]
    ax.imshow(make_overlay(image, maps['fused_vis_up'])); draw_boxes(ax, boxes)
    ax.set_title(f'{title}\np={maps["probability"]:.3f}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print('Gated weights:', method_maps['gated'].get('view_weights'))
print('Residual gates:', method_maps['residual'].get('residual_gates'))

## Main six-method comparison PDF

In [ ]:
MAIN_PDF_PATH = VIS_OUTPUT_DIR / 'compare_original_fc_gated_mean_residual_transformer_heatmaps.pdf'

METHOD_TITLES = {
    'fc': 'FC fusion',
    'gated': 'Gated fusion',
    'mean': 'Mean fusion',
    'residual': 'Residual fusion',
    'transformer': 'Transformer fusion',
}


def render_main_page(page_df, page_number, start_index):
    n = len(page_df)
    fig, axes = plt.subplots(n, 7, figsize=(24, 3.6*n), dpi=FIG_DPI, squeeze=False)
    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        orig = get_original_maps(row)

        axes[r,0].imshow(image); draw_boxes(axes[r,0], boxes)
        axes[r,0].set_title(
            f'{start_index+r+1:03d}. {row.image_id}\n{row.label} | GT={row.ground_truth}', fontsize=8
        )

        axes[r,1].imshow(make_overlay(image, orig['vis_up'])); draw_boxes(axes[r,1], boxes)
        axes[r,1].set_title(
            f'Original ALBEF\np={orig["probability"]:.4f} | m={orig["margin"]:.4f}', fontsize=8
        )

        for c, method in enumerate(['fc','gated','mean','residual','transformer'], start=2):
            maps = get_fusion_maps(row, method)
            axes[r,c].imshow(make_overlay(image, maps['fused_vis_up']))
            draw_boxes(axes[r,c], boxes)
            axes[r,c].set_title(
                f'{METHOD_TITLES[method]}\np={maps["probability"]:.4f} | m={maps["margin"]:.4f}', fontsize=8
            )

        for ax in axes[r]: ax.axis('off')

    fig.suptitle(
        f'Original ALBEF | FC | Gated | Mean | Residual | Transformer — page {page_number:02d}',
        fontsize=14, fontweight='bold', y=1.002,
    )
    plt.tight_layout()
    return fig

num_pages = math.ceil(len(gallery_df) / CASES_PER_PAGE)
print(f'Rendering {len(gallery_df)} canonical cases across {num_pages} pages')
with PdfPages(MAIN_PDF_PATH) as pdf:
    for page_number in range(num_pages):
        start = page_number * CASES_PER_PAGE
        stop = min(len(gallery_df), (page_number+1)*CASES_PER_PAGE)
        fig = render_main_page(gallery_df.iloc[start:stop], page_number+1, start)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
print('Saved:', MAIN_PDF_PATH)

## Branch-detail PDF

In [ ]:
BRANCH_PDF_PATH = VIS_OUTPUT_DIR / 'compare_fusion_branch_details.pdf'


def render_branch_page(page_df, method, page_number, start_index):
    n = len(page_df)
    fig, axes = plt.subplots(n, 6, figsize=(22, 3.6*n), dpi=FIG_DPI, squeeze=False)
    for r, row in enumerate(page_df.itertuples(index=False)):
        image = load_image(row.image_id)
        boxes = get_boxes(row.image_id, row.label, image.size)
        orig = get_original_maps(row)
        maps = get_fusion_maps(row, method)

        panels = [
            ('image', image),
            ('Original ALBEF', make_overlay(image, orig['vis_up'])),
            ('Original branch', make_overlay(image, maps['original_branch_vis_up'])),
            ('Lung branch', make_overlay(image, maps['lung_branch_vis_up'])),
            ('Heart branch', make_overlay(image, maps['heart_branch_vis_up'])),
            ('Composite fused', make_overlay(image, maps['fused_vis_up'])),
        ]
        for c, (title, panel) in enumerate(panels):
            axes[r,c].imshow(panel)
            draw_boxes(axes[r,c], boxes)
            if c == 0:
                axes[r,c].set_title(
                    f'{start_index+r+1:03d}. {row.image_id}\n{row.label} | GT={row.ground_truth}', fontsize=8
                )
            else:
                axes[r,c].set_title(title, fontsize=8)
            axes[r,c].axis('off')

    fig.suptitle(
        f'{METHOD_TITLES[method]} branch details — page {page_number:02d}',
        fontsize=14, fontweight='bold', y=1.002,
    )
    plt.tight_layout()
    return fig

if GENERATE_BRANCH_DETAIL_PDF:
    with PdfPages(BRANCH_PDF_PATH) as pdf:
        for method in ['fc','gated','mean','residual','transformer']:
            for page_number in range(math.ceil(len(gallery_df)/CASES_PER_PAGE)):
                start = page_number*CASES_PER_PAGE
                stop = min(len(gallery_df), (page_number+1)*CASES_PER_PAGE)
                fig = render_branch_page(
                    gallery_df.iloc[start:stop], method, page_number+1, start
                )
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
    print('Saved:', BRANCH_PDF_PATH)
else:
    print('Branch-detail PDF disabled.')

## Save order, diagnostics, and manifest

In [ ]:
GALLERY_ORDER_CSV = VIS_OUTPUT_DIR / 'canonical_100_comparison_gallery_order.csv'
DIAGNOSTICS_CSV = VIS_OUTPUT_DIR / 'canonical_100_comparison_diagnostics.csv'
MANIFEST_JSON = VIS_OUTPUT_DIR / 'canonical_100_comparison_manifest.json'

gallery_df.to_csv(GALLERY_ORDER_CSV, index=False)
comparison_df.to_csv(DIAGNOSTICS_CSV, index=False)

manifest = {
    'method_order': ['Original ALBEF','FC','Gated','Mean','Residual','Transformer'],
    'directories': {k: str(v) for k,v in METHOD_DIRS.items()},
    'biovil_manifest_csv': str(BIOVIL_MANIFEST_CSV),
    'target_labels': TARGET_LABELS,
    'n_cases': int(len(gallery_df)),
    'expected_pairs': EXPECTED_PAIR_COUNT,
    'expected_per_label': EXPECTED_PER_LABEL,
    'cases_per_page': CASES_PER_PAGE,
    'main_pdf': str(MAIN_PDF_PATH),
    'branch_pdf': str(BRANCH_PDF_PATH) if GENERATE_BRANCH_DETAIL_PDF else None,
    'gallery_order_csv': str(GALLERY_ORDER_CSV),
    'diagnostics_csv': str(DIAGNOSTICS_CSV),
}
with MANIFEST_JSON.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2)

print('Saved:', GALLERY_ORDER_CSV)
print('Saved:', DIAGNOSTICS_CSV)
print('Saved:', MANIFEST_JSON)
print('Saved:', MAIN_PDF_PATH)
if GENERATE_BRANCH_DETAIL_PDF:
    print('Saved:', BRANCH_PDF_PATH)

## Interpretation notes

- All fusion methods use the same **final fused ITC margin** as the backward target.
- `fused` is a **composite localization map**, created by summing raw positive original/lung/heart branch Grad-CAM evidence and normalizing once afterward.
- Gated-fusion weights and residual-fusion gates are already included in the gradient path; the CAMs are **not multiplied by these values again**.
- Transformer fusion self-attention operates across the three views at each fixed spatial token position. It is therefore not used as the spatial heatmap; localization is taken from the three ViT branches after backpropagation through the Transformer fusion module.
- Every main-PDF row corresponds to one of the exact canonical BioViL-T 100 image-label pairs.